# 파운데이션 모델 실습

**Foundation Model · 기반 모델**

넓은 데이터로 크게 사전학습해 여러 하위 과제에 적응시키는 대형 모델.

소재 분야에서 이해하기: 여러 원소계로 사전학습한 원자 모델을 특정 합금에 맞춰 미세조정한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 큰 데이터로 먼저 배우고, 작은 데이터에 맞추기

데이터가 많은 과제로 학습한 표현을 데이터가 적은 과제에 옮기는 효과를 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

def task(shift, n, seed):
    local = np.random.default_rng(seed)
    x = local.uniform(0, 1, (n, 3))
    y = (np.sin(3 * x[:, 0]) + 2 * x[:, 1] ** 2 - x[:, 2] + shift
         + local.normal(0, 0.05, n))
    return x, y

X_big, y_big = task(0.0, 4000, 1)      # 사전학습용 대규모 계산 데이터
X_few, y_few = task(0.4, 25, 2)        # 목표 과제의 소수 실험값
X_eval, y_eval = task(0.4, 500, 3)
print(X_big.shape, X_few.shape)

In [ ]:
pre = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0).fit(X_big, y_big)
scratch = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0).fit(X_few, y_few)

import copy
tuned = copy.deepcopy(pre)
tuned.set_params(max_iter=800, learning_rate_init=0.001)
for _ in range(40):
    tuned.partial_fit(X_few, y_few)

for name, model in [('사전학습만', pre), ('소수 데이터로 처음부터', scratch), ('사전학습 + 미세조정', tuned)]:
    print('%-22s MAE %.4f' % (name, mean_absolute_error(y_eval, model.predict(X_eval))))

## 2. 해석

사전학습 모델은 목표 과제의 오프셋을 모르지만 관계의 모양을 이미 알고 있어, 소수의 실험값만으로도
처음부터 학습하는 것보다 잘 맞습니다. 다만 사전학습 데이터와 목표가 너무 다르면 이득이 사라지거나
오히려 방해가 됩니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#foundation-model)을 여세요.